# IdentityDrift Access Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Identity Security
> **Data Sources:** SigninLogs, CommonSecurity_ID, DeviceProcessEvents, SecurityAlerts
> **Nodes:** 4 (Identity, Workload, Device, Alert)
> **Edges:** 6 (AccessTo, AccessedKubernetes, UsedDevice, ExecutedProcess, TriggeredAlert, DetectedOn)

Connects identities through access patterns, Kubernetes operations, device usage, process execution, and security alert correlations to enable full investigation of identity compromise, lateral movement, privilege escalation, and anomalous activities with threat intelligence enrichment.

In [8]:
# SDK compatibility check and configuration

# OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
TIME_WINDOW_DAYS = 1

# REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
workspace_name = "workspace_name"  # Replace with your Sentinel workspace name

import pkg_resources
import logging

version = pkg_resources.get_distribution('MicrosoftSentinelGraphProvider').version
print(f"MicrosoftSentinelGraphProvider: {version}" + f" using pool name : {spark.conf.get('spark.synapse.pool.name')}")
print(f"Running in: Region : {spark.conf.get('spark.cluster.region')}" + f" | Account id : {spark.conf.get('spark.pjs.account.id')}")

# Set the logging level for the entire package
logging.getLogger('sentinel_graph').setLevel(logging.INFO)
print(f"Logging level set to: {logging.getLevelName(logging.getLogger('sentinel_graph').level)} and above")

In [9]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
    col, lit, expr, count, first, lower, trim, when, concat, concat_ws,
    countDistinct, max as spark_max
)

lake_provider = MicrosoftSentinelProvider(spark)

# Helper function to extract PySpark DataFrame from CustomDataFrame wrapper
def to_spark_df(df):
    return df.df if hasattr(df, 'df') else df

print(f"Reading data from workspace: {workspace_name}")
print(f"Lookback period: {TIME_WINDOW_DAYS} days")

In [12]:
# =============================================================================
# Read Source Tables from Sentinel
# =============================================================================

# =============================================================================
# Read SigninLogs (Azure AD authentication events)
# =============================================================================
signin_logs_df = (
    to_spark_df(lake_provider.read_table("SigninLogs_ID_KQL_CL", workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('UserPrincipalName').isNotNull())
    .select(
        "TimeGenerated", "UserPrincipalName", "AppDisplayName", "RiskLevelAggregated",
        "ConditionalAccessStatus", "IPAddress"
    )
).persist()

# =============================================================================
# Read CommonSecurity_ID_Logs (Kubernetes/Infrastructure access events)
# =============================================================================
common_security_df = (
    to_spark_df(lake_provider.read_table("CommonSecurity_ID_KQL_CL", workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('SourceUserName').isNotNull())
    .select(
        "TimeGenerated", "SourceUserName", "DestinationHostName", "DeviceCustomString1",
        "SourceIP", "DeviceCustomString1Label"
    )
).persist()

# =============================================================================
# Read DeviceProcessEvents (Process execution on endpoints)
# =============================================================================
device_process_df = (
    to_spark_df(lake_provider.read_table("DeviceProcessEvents_KQL_CL", workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('ActionType') == 'ProcessCreated')
    .select(
        "TimeGenerated", "DeviceId", "DeviceName", "AccountName", "FileName",
        "ProcessCommandLine", "ProcessIntegrityLevel", "FolderPath", "ProcessId", "AccountDomain"
    )
).persist()

# =============================================================================
# Read SecurityAlerts (Security detections and incidents)
# =============================================================================
security_alerts_df = (
    to_spark_df(lake_provider.read_table("SecurityAlerts_KQL_CL", workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('AlertName').isNotNull())
    .select(
        "TimeGenerated", "AlertName", "DisplayName", "AlertSeverity", "AlertType",
        "ConfidenceLevel", "IsIncident", "CompromisedEntity", "Description"
    )
).persist()

# Validation counts
print(f"SigninLogs: {signin_logs_df.count()}")
print(f"CommonSecurity_ID_Logs: {common_security_df.count()}")
print(f"DeviceProcessEvents: {device_process_df.count()}")
print(f"SecurityAlerts: {security_alerts_df.count()}")

## Node Dataframes

Building the 4 node types from source tables.

In [14]:
# =============================================================================
# NODE DATAFRAMES (4 types)
# =============================================================================

# --- Identity nodes (from SigninLogs and CommonSecurity_ID_Logs) ---
identity_signin = signin_logs_df.select(
    col("UserPrincipalName").alias("IdentityId"),
    col("RiskLevelAggregated")
).distinct()

identity_common_sec = common_security_df.select(
    col("SourceUserName").alias("IdentityId"),
    lit(None).cast("string").alias("RiskLevelAggregated")
).distinct()

identity_node_df = (
    identity_signin.union(identity_common_sec)
    .distinct()
    .filter(col("IdentityId").isNotNull())
    .withColumn("nodeType", lit("Identity"))
)

# --- Workload nodes (from SigninLogs: applications + CommonSecurity: infrastructure) ---
workload_signin = signin_logs_df.select(
    col("AppDisplayName").alias("WorkloadId"),
    col("AppDisplayName").alias("WorkloadName"),
    lit("Application").alias("WorkloadType")
).distinct()

workload_infra = common_security_df.select(
    col("DestinationHostName").alias("WorkloadId"),
    col("DestinationHostName").alias("WorkloadName"),
    lit("Infrastructure").alias("WorkloadType")
).distinct()

workload_node_df = (
    workload_signin.union(workload_infra)
    .distinct()
    .filter(col("WorkloadId").isNotNull())
    .withColumn("nodeType", lit("Workload"))
)

# --- Device nodes (from DeviceProcessEvents) ---
device_node_df = (
    device_process_df
    .select(col("DeviceName").alias("DeviceId"), "DeviceName")
    .distinct()
    .filter(col("DeviceId").isNotNull())
    .withColumn("nodeType", lit("Device"))
)

# --- Alert nodes (from SecurityAlerts) ---
alert_node_df = (
    security_alerts_df
    .select(
        col("AlertName").alias("AlertId"),
        "DisplayName", "AlertSeverity", "AlertType",
        "ConfidenceLevel", "IsIncident"
    )
    .distinct()
    .filter(col("AlertId").isNotNull())
    .withColumn("nodeType", lit("Alert"))
)

# Validation counts
print(f"Nodes - Identity: {identity_node_df.count()}, Workload: {workload_node_df.count()}")
print(f"Nodes - Device: {device_node_df.count()}, Alert: {alert_node_df.count()}")

In [ ]:
# =============================================================================
# EDGE DATAFRAMES (6 types)
# =============================================================================

# --- AccessTo: Identity -> Workload (from SigninLogs) ---
access_to_edge_df = (
    signin_logs_df
    .select(
        col("UserPrincipalName").alias("IdentityId"),
        col("AppDisplayName").alias("WorkloadId"),
        "RiskLevelAggregated", "ConditionalAccessStatus", "TimeGenerated"
    )
    .filter(col("IdentityId").isNotNull() & col("WorkloadId").isNotNull())
    .groupBy("IdentityId", "WorkloadId")
    .agg(
        spark_max("TimeGenerated").alias("LastActivity"),
        count("*").alias("AccessCount"),
        first("RiskLevelAggregated").alias("RiskLevel")
    )
    .withColumn("edgeType", lit("AccessTo"))
    .withColumn("EdgeKey", concat_ws("_", col("IdentityId"), col("WorkloadId")))
)

# --- AccessedInfrastructure: Identity -> Workload (from CommonSecurity_ID_Logs) ---
accessed_infra_edge_df = (
    common_security_df
    .select(
        col("SourceUserName").alias("IdentityId"),
        col("DestinationHostName").alias("WorkloadId"),
        "DeviceCustomString1", "SourceIP", "TimeGenerated"
    )
    .filter(col("IdentityId").isNotNull() & col("WorkloadId").isNotNull())
    .groupBy("IdentityId", "WorkloadId")
    .agg(
        spark_max("TimeGenerated").alias("LastActivity"),
        count("*").alias("EventCount"),
        countDistinct("DeviceCustomString1").alias("ActivityTypes"),
        countDistinct("SourceIP").alias("SourceIPCount")
    )
    .withColumn("edgeType", lit("AccessedInfrastructure"))
    .withColumn("EdgeKey", concat_ws("_", col("IdentityId"), col("WorkloadId")))
)

# --- UsedDevice: Identity -> Device ---
used_device_edge_df = (
    device_process_df
    .select(
        col("AccountName").alias("IdentityId"),
        col("DeviceName").alias("DeviceId"),
        "TimeGenerated"
    )
    .filter(col("IdentityId").isNotNull() & col("DeviceId").isNotNull())
    .groupBy("IdentityId", "DeviceId")
    .agg(
        spark_max("TimeGenerated").alias("LastActivity"),
        count("*").alias("ProcessCount")
    )
    .withColumn("edgeType", lit("UsedDevice"))
    .withColumn("EdgeKey", concat_ws("_", col("IdentityId"), col("DeviceId")))
)

# --- ExecutedProcess: Identity -> Process ---
executed_process_edge_df = (
    device_process_df
    .withColumn("ProcessId", concat_ws("_", col("DeviceName"), col("FileName"), col("TimeGenerated")))
    .select(
        col("AccountName").alias("IdentityId"),
        "ProcessId",
        "FileName", "ProcessCommandLine", "ProcessIntegrityLevel", "TimeGenerated"
    )
    .filter(col("IdentityId").isNotNull())
    .groupBy("IdentityId", "ProcessId")
    .agg(
        first("FileName").alias("ProcessName"),
        first("ProcessCommandLine").alias("CommandLine"),
        first("ProcessIntegrityLevel").alias("IntegrityLevel"),
        spark_max("TimeGenerated").alias("LastExecution"),
        count("*").alias("ExecutionCount")
    )
    .withColumn("edgeType", lit("ExecutedProcess"))
    .withColumn("EdgeKey", concat_ws("_", col("IdentityId"), col("ProcessId")))
)

# --- TriggeredAlert: Identity -> Alert (extracted from alert CompromisedEntity) ---
triggered_alert_edge_df = (
    security_alerts_df
    .select(
        col("CompromisedEntity").alias("IdentityId"),
        col("AlertName").alias("AlertId"),
        "AlertSeverity", "ConfidenceLevel", "TimeGenerated"
    )
    .filter(col("IdentityId").isNotNull() & col("AlertId").isNotNull())
    .groupBy("IdentityId", "AlertId")
    .agg(
        spark_max("TimeGenerated").alias("LastTrigger"),
        count("*").alias("TriggerCount"),
        first("AlertSeverity").alias("Severity")
    )
    .withColumn("edgeType", lit("TriggeredAlert"))
    .withColumn("EdgeKey", concat_ws("_", col("IdentityId"), col("AlertId")))
)

# --- DetectedOn: Alert -> Workload ---
detected_on_edge_df = (
    security_alerts_df
    .select(
        col("AlertName").alias("AlertId"),
        col("CompromisedEntity").alias("WorkloadId"),
        "AlertSeverity", "Description", "TimeGenerated"
    )
    .filter(col("AlertId").isNotNull() & col("WorkloadId").isNotNull())
    .groupBy("AlertId", "WorkloadId")
    .agg(
        spark_max("TimeGenerated").alias("LastDetection"),
        count("*").alias("DetectionCount"),
        first("AlertSeverity").alias("Severity"),
        first("Description").alias("Description")
    )
    .withColumn("edgeType", lit("DetectedOn"))
    .withColumn("EdgeKey", concat_ws("_", col("AlertId"), col("WorkloadId")))
)

# Validation counts
print(f"Edges - AccessTo: {access_to_edge_df.count()}, AccessedInfrastructure: {accessed_infra_edge_df.count()}")
print(f"Edges - UsedDevice: {used_device_edge_df.count()}, ExecutedProcess: {executed_process_edge_df.count()}")
print(f"Edges - TriggeredAlert: {triggered_alert_edge_df.count()}, DetectedOn: {detected_on_edge_df.count()}")

In [19]:
# =============================================================================
# GRAPH SPECIFICATION & ASSEMBLY
# =============================================================================

# Build Graph Specification using fluent API
identity_drift_graph_spec = (GraphSpecBuilder.start()

    # ===================== NODES (4 types) =====================

    .add_node("Identity")
    .from_dataframe(identity_node_df)
    .with_columns(
        "IdentityId", "RiskLevelAggregated", "nodeType",
        key="IdentityId", display="IdentityId"
    )

    .add_node("Workload")
    .from_dataframe(workload_node_df)
    .with_columns(
        "WorkloadId", "WorkloadName", "WorkloadType", "nodeType",
        key="WorkloadId", display="WorkloadName"
    )

    .add_node("Device")
    .from_dataframe(device_node_df)
    .with_columns(
        "DeviceId", "DeviceName", "nodeType",
        key="DeviceId", display="DeviceName"
    )

    .add_node("Alert")
    .from_dataframe(alert_node_df)
    .with_columns(
        "AlertId", "DisplayName", "AlertSeverity", "AlertType",
        "ConfidenceLevel", "IsIncident", "nodeType",
        key="AlertId", display="DisplayName"
    )

    # ===================== EDGES (6 types) =====================

    .add_edge("AccessTo")
    .from_dataframe(access_to_edge_df)
    .source(id_column="IdentityId", node_type="Identity")
    .target(id_column="WorkloadId", node_type="Workload")
    .with_columns(
        "RiskLevel", "AccessCount", "LastActivity", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("AccessedInfrastructure")
    .from_dataframe(accessed_infra_edge_df)
    .source(id_column="IdentityId", node_type="Identity")
    .target(id_column="WorkloadId", node_type="Workload")
    .with_columns(
        "EventCount", "ActivityTypes", "SourceIPCount", "LastActivity", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("UsedDevice")
    .from_dataframe(used_device_edge_df)
    .source(id_column="IdentityId", node_type="Identity")
    .target(id_column="DeviceId", node_type="Device")
    .with_columns(
        "ProcessCount", "LastActivity", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("ExecutedProcess")
    .from_dataframe(executed_process_edge_df)
    .source(id_column="IdentityId", node_type="Identity")
    .target(id_column="ProcessId", node_type="Device")
    .with_columns(
        "ProcessName", "CommandLine", "IntegrityLevel", "ExecutionCount",
        "LastExecution", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("TriggeredAlert")
    .from_dataframe(triggered_alert_edge_df)
    .source(id_column="IdentityId", node_type="Identity")
    .target(id_column="AlertId", node_type="Alert")
    .with_columns(
        "Severity", "TriggerCount", "LastTrigger", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("DetectedOn")
    .from_dataframe(detected_on_edge_df)
    .source(id_column="AlertId", node_type="Alert")
    .target(id_column="WorkloadId", node_type="Workload")
    .with_columns(
        "Severity", "DetectionCount", "LastDetection", "edgeType", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

).done()

print("Graph specification created successfully!")


In [20]:
# Check the schema of the graph spec to ensure it is correct
identity_drift_graph_spec.show_schema()

In [36]:
try:
    print("Building graph from specification...")
    identity_drift_graph = Graph.build(identity_drift_graph_spec)
    
    print("Graph build completed successfully!")
    print(f"Graph Name: {identity_drift_graph.name}")
    print(f"Nodes: {identity_drift_graph.nodes.count()}")
    print(f"Edges: {identity_drift_graph.edges.count()}")
    
except Exception as e:
    print(f"Error building graph: {e}")
    import traceback
    traceback.print_exc()